# 003d — PMC as a symmetry plane

A perfect magnetic conductor (PMC) can replace a mirror plane when the tangential electric field is **even** and the tangential magnetic field is **odd** across that plane. Here a standing wave in a cavity spanning $-L \le z \le L$ has PEC walls at both ends. Its mirror plane is $z=0$, so the half cavity $0 \le z \le L$ uses a PMC face at $z=0$ and a PEC face at $z=L$.

We compare both simulations on the same physical mesh and timestep. Periodic transverse faces isolate a one-dimensional $E_x$/$H_y$ mode. The initial conditions are

$$E_x(z,0)=E_0\cos\!\left(\frac{\pi z}{2L}ight), \qquad H_y(z,0)=0.$$

The electric field is even, vanishes at the PEC walls, and has zero normal derivative at the PMC plane. The magnetic field later becomes odd, so its tangential value vanishes at the mirror plane. This symmetry reduction is valid only when the geometry, materials, and excitation share the same parity.

## Set up full and half cavities

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML
from matplotlib.animation import FuncAnimation

from wakis import GridFIT3D, SolverFIT3D

half_length = 0.20  # m
half_cells = 80
dt = 4e-12  # s
nsteps = 400
transverse = np.linspace(0.0, 0.03, 4)
full_z = np.linspace(-half_length, half_length, 2 * half_cells + 1)
half_z = np.linspace(0.0, half_length, half_cells + 1)

full_grid = GridFIT3D(x=transverse, y=transverse, z=full_z, verbose=0)
half_grid = GridFIT3D(x=transverse, y=transverse, z=half_z, verbose=0)
full_solver = SolverFIT3D(
    full_grid,
    bc_low=["periodic", "periodic", "pec"],
    bc_high=["periodic", "periodic", "pec"],
    dt=dt,
    verbose=0,
)
half_solver = SolverFIT3D(
    half_grid,
    bc_low=["periodic", "periodic", "pmc"],
    bc_high=["periodic", "periodic", "pec"],
    dt=dt,
    verbose=0,
)

# Ex occupies physical x edges; the high x storage plane is inactive.
full_profile = np.cos(np.pi * full_solver.z / (2 * half_length))
half_profile = np.cos(np.pi * half_solver.z / (2 * half_length))
full_solver.E.to_matrix("x")[:-1] = full_profile[None, None, :]
half_solver.E.to_matrix("x")[:-1] = half_profile[None, None, :]

print(f"Longitudinal stored nodes: full {full_grid.Nz}, half {half_grid.Nz}")

## Evolve and compare mirror parity

We reconstruct the missing negative half from the PMC solution. $E_x$ should match the full cavity with even parity. $H_y$ lives in the cells on either side of the mirror plane and should have opposite signs there; no magnetic sample needs to sit exactly on the plane.

In [ ]:
times = np.arange(nsteps + 1) * dt
frame_stride = 10  # one animation frame every 40 ps
frame_times = times[::frame_stride]
full_frames = np.empty((len(frame_times), full_z.size))
half_frames = np.empty((len(frame_times), half_z.size))
full_center = np.empty(nsteps + 1)
half_center = np.empty(nsteps + 1)
electric_error = np.empty(nsteps + 1)
magnetic_error = np.empty(nsteps + 1)

for step in range(nsteps + 1):
    if step:
        full_solver.one_step()
        half_solver.one_step()

    full_e = full_solver.E.to_matrix("x")[0, 0, :]
    half_e = half_solver.E.to_matrix("x")[0, 0, :]
    mirrored_e = np.r_[half_e[:0:-1], half_e]
    electric_error[step] = np.max(np.abs(full_e - mirrored_e))
    full_center[step] = full_e[half_cells]
    half_center[step] = half_e[0]
    if step % frame_stride == 0:
        full_frames[step // frame_stride] = full_e
        half_frames[step // frame_stride] = half_e

    full_h = full_solver.H.to_matrix("y")[0, 0, :]
    half_h = half_solver.H.to_matrix("y")[0, 0, :]
    magnetic_error[step] = max(
        np.max(np.abs(full_h[:half_cells] + half_h[:half_cells][::-1])),
        np.max(np.abs(full_h[half_cells : 2 * half_cells] - half_h[:half_cells])),
    )

print(f"Largest Ex mirror difference: {electric_error.max():.2e} V/m")
print(f"Largest Hy mirror difference: {magnetic_error.max():.2e} A/m")

## Animated field comparison

The left panel shows the full cavity. The right panel shows the simulated PMC half in orange and its mirrored copy in red, completing the wave across the symmetry plane. Both panels keep a fixed electric-field scale as the standing wave changes sign.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
(full_line,) = axes[0].plot(full_z, full_frames[0], label="Full cavity")
(half_line,) = axes[1].plot(
    half_z, half_frames[0], color="tab:orange", label="Simulated PMC half"
)
(mirror_line,) = axes[1].plot(
    -half_z[::-1], half_frames[0, ::-1], color="red", label="Mirrored half"
)
for ax in axes:
    ax.axvline(0.0, color="0.5", linewidth=1)
    ax.set_xlim(-half_length, half_length)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xlabel("z [m]")
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel(r"$E_x$ [V/m]")
axes[0].set_title("Full PEC cavity")
axes[1].set_title("PMC symmetry half")
title = fig.suptitle("")
fig.tight_layout()


def update_frame(frame):
    full_line.set_ydata(full_frames[frame])
    half_line.set_ydata(half_frames[frame])
    mirror_line.set_ydata(half_frames[frame, ::-1])
    title.set_text(f"Standing-wave fields at {frame_times[frame] * 1e9:.2f} ns")
    return full_line, half_line, mirror_line, title


animation = FuncAnimation(fig, update_frame, frames=len(frame_times), interval=100)
plt.close(fig)
HTML(animation.to_jshtml())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(times * 1e9, full_center, label="Full cavity at z = 0")
ax.plot(times * 1e9, half_center, "--", label="PMC half at z = 0")
ax.set(xlabel="Time [ns]", ylabel=r"$E_x(0,t)$ [V/m]")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()

The half cavity reproduces the full cavity while storing about half as many longitudinal samples. Use a PMC symmetry face when tangential $E$ is even and tangential $H$ is odd across the plane. If tangential $E$ is odd, the corresponding electric symmetry face is PEC instead.